## Preparación de los datos

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())  # Verifica si tienes acceso a GPU



2.5.1
False


In [2]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset

class GalaxyDataset(Dataset):
    def __init__(self, input_dir, target_dir):
        self.input_files = sorted([os.path.join(input_dir, f) for f in os.listdir(input_dir) if f.endswith('.npy')])
        self.target_files = sorted([os.path.join(target_dir, f) for f in os.listdir(target_dir) if f.endswith('.npy')])

    def __len__(self):
        return len(self.input_files)

    def __getitem__(self, idx):
        input_image = np.load(self.input_files[idx])
        target_image = np.load(self.target_files[idx])
        return torch.tensor(input_image, dtype=torch.float32).unsqueeze(0), torch.tensor(target_image, dtype=torch.float32).unsqueeze(0)


## Definir el modelo U-Net

In [3]:
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, features=64):
        super(UNet, self).__init__()
        self.encoder1 = nn.Sequential(
            nn.Conv2d(in_channels, features, 3, padding=1),
            nn.LeakyReLU(),
            nn.Conv2d(features, features, 3, padding=1),
            nn.LeakyReLU()
        )
        self.pool = nn.MaxPool2d(2)
        self.decoder1 = nn.Sequential(
            nn.Conv2d(features, features, 3, padding=1),
            nn.LeakyReLU(),
            nn.Conv2d(features, out_channels, 3, padding=1),
            nn.LeakyReLU()
        )

    def forward(self, x, t):
        x1 = self.encoder1(x)
        x2 = self.pool(x1)
        x3 = F.interpolate(x2, scale_factor=2, mode='nearest')
        x4 = self.decoder1(x3)
        return x4


## Implementar el proceso de difusión

In [4]:
import torch

class Diffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02):
        self.timesteps = timesteps
        self.betas = torch.linspace(beta_start, beta_end, timesteps)
        self.alphas = 1. - self.betas
        self.alpha_hat = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)

        
        sqrt_alpha_hat = self.alpha_hat[t].view(-1, 1, 1, 1).to(x0.device)
        sqrt_one_minus_alpha_hat = (1 - self.alpha_hat[t]).view(-1, 1, 1, 1).to(x0.device)

        return sqrt_alpha_hat * x0 + sqrt_one_minus_alpha_hat * noise, noise



## Entrenar el modelo

In [5]:
import torch
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_

# Configuraciones
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
epochs = 10
batch_size = 8
learning_rate = 1e-5


# Datos
dataset = GalaxyDataset('../data/Resized64/dud/', '../data/Resized64/acs/')
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Modelo y optimizador
model = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
diffusion = Diffusion()

# Entrenamiento
for epoch in range(epochs):
    for step, (x, y) in enumerate(dataloader):
        x = x.to(device)
        y = y.to(device)
        t = torch.randint(0, diffusion.timesteps, (x.size(0),), device=device).long()
        noisy_y, noise = diffusion.add_noise(y, t)
        output = model(noisy_y, t)
        loss = F.mse_loss(output, noise)
        optimizer.zero_grad()
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if step % 100 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss: {loss.item():.6f}")
    
    print(f"✅ Epoch {epoch+1} completado")



Epoch 1 | Step 0 | Loss: 0.993124
Epoch 1 | Step 100 | Loss: 0.966868
Epoch 1 | Step 200 | Loss: 0.915701
Epoch 1 | Step 300 | Loss: 0.890783
Epoch 1 | Step 400 | Loss: 0.832181
Epoch 1 | Step 500 | Loss: nan


KeyboardInterrupt: 

## Generar imágenes

In [ ]:
def sample(model, diffusion, shape):
    model.eval()
    with torch.no_grad():
        x = torch.randn(shape).to(device)
        for t in reversed(range(diffusion.timesteps)):
            t_tensor = torch.full((shape[0],), t, device=device, dtype=torch.long)
            predicted_noise = model(x, t_tensor)
            alpha = diffusion.alphas[t]
            alpha_hat = diffusion.alpha_hat[t]
            beta = diffusion.betas[t]
            x = (1 / alpha ** 0.5) * (x - ((1 - alpha) / (1 - alpha_hat) ** 0.5) * predicted_noise)
            if t > 0:
                noise = torch.randn_like(x)
                x += beta ** 0.5 * noise
    return x
